# 📑 Notebook 1 — Chunking Hierárquico

Implementa chunking hierárquico estrutural conforme a Etapa 6 do pipeline:

- **Filho** (≤ 256 tokens): vai para o índice vetorial
- **Pai** (≤ 512 tokens): filho_anterior + filho_atual + filho_seguinte → enviado ao LLM
- **Estratégia por tipo:** texto_integral (Art./§/Inciso), voto (tamanho+overlap), nota_tecnica (seções), anexo (cabeçalho repetido), decisao (seções fixas)
- **Limites por tipo:** protege o índice de documentos gigantes (glossários MCSD)

**Tempo estimado:** ~30-40 minutos (não precisa GPU)

In [ ]:
# ============================================================
# CÉLULA 1 — Instalação
# ============================================================
%%capture
!pip install datasets transformers tqdm pyarrow huggingface_hub python-dotenv

In [ ]:
# ============================================================
# CÉLULA 2 — Imports
# ============================================================
import os
import re
import gc
import json
import pickle
import pandas as pd
import numpy as np
from tqdm.notebook import tqdm
from datasets import load_dataset
from transformers import AutoTokenizer

OUTPUT_DIR = '/content/aneel_rag'
os.makedirs(OUTPUT_DIR, exist_ok=True)
print('✅ Imports OK')

In [ ]:
# ============================================================
# CÉLULA 2-B — Verificar artefatos existentes no HF
# ============================================================
from huggingface_hub import list_repo_files, hf_hub_download
from dotenv import load_dotenv
import os

load_dotenv()
HF_TOKEN = os.environ.get('HF_TOKEN', '')
HF_REPO  = 'JvPetas/aneel-legislacao'

print('Verificando artefatos no Hugging Face...')
try:
    arquivos_hf = list(list_repo_files(HF_REPO, repo_type='dataset', token=HF_TOKEN))
    parquet_existe = 'chunks_hierarquicos.parquet' in arquivos_hf
except Exception as e:
    print(f'Nao foi possivel verificar HF: {e}')
    parquet_existe = False

if parquet_existe:
    print('chunks_hierarquicos.parquet encontrado no HF!')
    print('   Baixando...')
    hf_hub_download(
        repo_id=HF_REPO,
        filename='chunks_hierarquicos.parquet',
        repo_type='dataset',
        local_dir=OUTPUT_DIR,
        token=HF_TOKEN,
    )
    print('Download concluido. Chunking nao e necessario.')
    print('   Pode ir direto para o Notebook 2.')
    SKIP_CHUNKING = True
else:
    print('Parquet nao encontrado no HF — chunking necessario.')
    SKIP_CHUNKING = False

In [ ]:
# ============================================================
# CÉLULA 3 — Configurações do Chunking
# ============================================================

# Tamanhos (em tokens, calculados pelo tokenizer do e5)
CHILD_MAX_TOKENS = 256   # Chunk filho — granularidade do índice
PARENT_MAX_TOKENS = 512  # Chunk pai — contexto enviado ao LLM
OVERLAP_TOKENS    = 50   # Overlap entre chunks consecutivos (votos)

# Filtros de qualidade
QUALIDADE_MINIMA = 0.7
MIN_CHARS_DOC    = 200
MIN_CHARS_CHUNK  = 100

# Limites máximos de chunks por documento (proteção contra glossários enormes)
# Conforme decisão do PDF: glossários MCSD geram até 2.177 chunks → contaminação
LIMITES_CHUNKS = {
    'texto_integral': 120,
    'voto':           80,
    'nota_tecnica':   100,
    'anexo':          150,  # anexos podem ter tabelas legítimas
    'decisao':        50,
    'outro':          40,
}

# Tokenizer do modelo de embedding (para contar tokens corretamente)
EMBEDDING_MODEL = 'intfloat/multilingual-e5-large-instruct'

print('✅ Configs definidas')
print(f'   Filho: {CHILD_MAX_TOKENS} tokens | Pai: {PARENT_MAX_TOKENS} tokens')
print(f'   Tokenizer: {EMBEDDING_MODEL}')

In [ ]:
# ============================================================
# CÉLULA 4 — Carrega tokenizer
# ============================================================
print(f'📥 Carregando tokenizer: {EMBEDDING_MODEL}')
tokenizer = AutoTokenizer.from_pretrained(EMBEDDING_MODEL)

def count_tokens(text: str) -> int:
    """Conta tokens reais usando o tokenizer do e5."""
    return len(tokenizer.encode(text, add_special_tokens=False))

# Sanity check
exemplo = 'Art. 1º Esta Resolução estabelece os procedimentos para revisão tarifária.'
print(f"✅ Tokenizer pronto. Exemplo: '{exemplo}'")
print(f"   {count_tokens(exemplo)} tokens / {len(exemplo)} chars (ratio: {len(exemplo)/count_tokens(exemplo):.2f})")

In [ ]:
# ============================================================
# CÉLULA 5 — Carrega dataset HF
# ============================================================
print('📥 Carregando JvPetas/aneel-legislacao...')
ds = load_dataset('JvPetas/aneel-legislacao', split='train')
df = ds.to_pandas()
print(f'✅ {len(df):,} documentos')

# Filtragem inicial
before = len(df)
df = df[
    (df['qualidade_score'] >= QUALIDADE_MINIMA) &
    (df['texto'].notna()) &
    (df['texto'].str.len() >= MIN_CHARS_DOC)
].reset_index(drop=True)
print(f'✅ Após filtros de qualidade: {len(df):,} ({len(df)/before*100:.1f}%)')
print(f'\n📊 Distribuição por tipo:')
print(df['tipo_documento'].value_counts().to_string())

In [ ]:
# ============================================================
# CÉLULA 6 — Funções auxiliares de chunking
# ============================================================

def split_by_tokens(text: str, max_tokens: int, overlap: int = 0) -> list:
    """
    Divide texto por tokens com overlap. Usado quando não há estrutura clara.
    Tenta quebrar em fronteiras naturais (parágrafo > sentença > espaço).
    """
    text = text.strip()
    if not text:
        return []

    # Tokeniza uma vez
    tokens = tokenizer.encode(text, add_special_tokens=False)
    if len(tokens) <= max_tokens:
        return [text]

    chunks = []
    step = max_tokens - overlap
    for i in range(0, len(tokens), step):
        window = tokens[i : i + max_tokens]
        chunks.append(tokenizer.decode(window, skip_special_tokens=True).strip())
        if i + max_tokens >= len(tokens):
            break
    return [c for c in chunks if len(c) >= MIN_CHARS_CHUNK]


def join_until_limit(units: list, max_tokens: int) -> list:
    """
    Junta unidades textuais (artigos, parágrafos, seções) em chunks
    sem ultrapassar max_tokens. Unidades muito grandes são re-divididas.
    """
    chunks, current, current_tokens = [], [], 0

    for unit in units:
        unit = unit.strip()
        if not unit:
            continue

        unit_tokens = count_tokens(unit)

        # Unidade gigante: split forçado por tokens
        if unit_tokens > max_tokens:
            if current:
                chunks.append('\n\n'.join(current))
                current, current_tokens = [], 0
            chunks.extend(split_by_tokens(unit, max_tokens))
            continue

        # Cabe no chunk atual
        if current_tokens + unit_tokens <= max_tokens:
            current.append(unit)
            current_tokens += unit_tokens
        else:
            if current:
                chunks.append('\n\n'.join(current))
            current, current_tokens = [unit], unit_tokens

    if current:
        chunks.append('\n\n'.join(current))
    return [c for c in chunks if len(c) >= MIN_CHARS_CHUNK]


def extrair_contexto_juridico(text: str, ato_id: str, titulo: str) -> str:
    """
    Extrai marcador de localização jurídica do trecho (Art. X, §, Inciso).
    Usado como prefixo no chunk filho para enriquecer o embedding.
    """
    contexto_partes = []

    # Match em ordem decrescente de prioridade
    art_match = re.search(r'\bArt\.\s*(\d+)[ºo°]?', text)
    if art_match:
        contexto_partes.append(f'Art. {art_match.group(1)}')

    par_match = re.search(r'§\s*(\d+)[ºo°]?', text)
    if par_match:
        contexto_partes.append(f'§{par_match.group(1)}')
    elif 'Parágrafo único' in text or 'parágrafo único' in text:
        contexto_partes.append('Parágrafo único')

    if not contexto_partes:
        return titulo or ato_id
    return f"{titulo or ato_id} - {', '.join(contexto_partes)}"


print('✅ Funções auxiliares definidas')

In [ ]:
# ============================================================
# CÉLULA 7 — Estratégias de chunking por tipo de documento
# ============================================================

# Padrões de divisão estrutural para textos legais brasileiros
PADRAO_ARTIGO   = re.compile(r'(?=\bArt\.\s*\d+[ºo°]?)')
PADRAO_PARAGRAFO = re.compile(r'(?=§\s*\d+[ºo°]?|\bParágrafo único\b)')
PADRAO_SECAO_NT  = re.compile(r'(?=\n\s*\d+\.\s*[A-Z])')  # "1. INTRODUÇÃO" "2. ANÁLISE"
PADRAO_QUEBRA    = re.compile(r'\n\s*\n+')


def chunk_texto_integral(text: str) -> list:
    """
    Resoluções, normativos: divide por Art. e §.
    Cada artigo (ou conjunto) vira um chunk filho ≤ CHILD_MAX_TOKENS.
    """
    # Tenta dividir por artigos
    artigos = PADRAO_ARTIGO.split(text)
    artigos = [a.strip() for a in artigos if a.strip()]

    # Se não tem artigos suficientes, divide por parágrafos
    if len(artigos) < 2:
        artigos = PADRAO_PARAGRAFO.split(text)
        artigos = [a.strip() for a in artigos if a.strip()]

    # Último fallback: parágrafos do texto
    if len(artigos) < 2:
        artigos = PADRAO_QUEBRA.split(text)
        artigos = [a.strip() for a in artigos if a.strip()]

    return join_until_limit(artigos, CHILD_MAX_TOKENS)


def chunk_voto(text: str) -> list:
    """
    Votos: peças argumentativas com fluxo discursivo.
    Divide por tamanho com overlap (preserva contexto).
    """
    return split_by_tokens(text, CHILD_MAX_TOKENS, overlap=OVERLAP_TOKENS)


def chunk_nota_tecnica(text: str) -> list:
    """
    Notas técnicas: seções numeradas (1. Introdução, 2. Análise...).
    Mantém tabelas inteiras quando possível.
    """
    secoes = PADRAO_SECAO_NT.split(text)
    secoes = [s.strip() for s in secoes if s.strip()]

    if len(secoes) < 2:
        secoes = PADRAO_QUEBRA.split(text)
        secoes = [s.strip() for s in secoes if s.strip()]

    return join_until_limit(secoes, CHILD_MAX_TOKENS)


def chunk_anexo(text: str, titulo: str = '') -> list:
    """
    Anexos: tabelas e dados estruturados. Cabeçalho repetido em cada chunk
    para preservar contexto após divisão.
    """
    # Pega as primeiras linhas como cabeçalho contextual
    linhas = text.split('\n')
    cabecalho = '\n'.join(linhas[:3]).strip()
    cabecalho_tokens = count_tokens(cabecalho)

    if cabecalho_tokens > 50:
        cabecalho = titulo or ''
        cabecalho_tokens = count_tokens(cabecalho)

    body = '\n'.join(linhas[3:]) if cabecalho_tokens > 0 else text

    # Espaço útil considerando o cabeçalho repetido
    available = max(CHILD_MAX_TOKENS - cabecalho_tokens - 5, 100)
    chunks = split_by_tokens(body, available, overlap=20)

    if cabecalho:
        return [f'{cabecalho}\n\n{c}' for c in chunks]
    return chunks


def chunk_decisao(text: str) -> list:
    """
    Decisões: estrutura formal (Relatório, Voto, Decisão).
    Divide por seções e por parágrafos.
    """
    paragraphs = PADRAO_QUEBRA.split(text)
    paragraphs = [p.strip() for p in paragraphs if p.strip()]
    return join_until_limit(paragraphs, CHILD_MAX_TOKENS)


def chunk_documento(row: dict) -> list:
    """Roteador: chama a estratégia certa conforme tipo_documento."""
    tipo = row['tipo_documento']
    text = row['texto']
    titulo = str(row.get('titulo') or '')

    if tipo == 'texto_integral':
        return chunk_texto_integral(text)
    elif tipo == 'voto':
        return chunk_voto(text)
    elif tipo == 'nota_tecnica':
        return chunk_nota_tecnica(text)
    elif tipo == 'anexo':
        return chunk_anexo(text, titulo)
    elif tipo == 'decisao':
        return chunk_decisao(text)
    else:
        return split_by_tokens(text, CHILD_MAX_TOKENS, overlap=OVERLAP_TOKENS)


print('✅ Estratégias de chunking definidas')

In [ ]:
# ============================================================
# CÉLULA 8 — Teste rápido das estratégias (em 5 docs por tipo)
# ============================================================
print('🧪 Testando estratégias em amostra...\n')

for tipo in df['tipo_documento'].unique():
    sample = df[df['tipo_documento'] == tipo].head(3)
    if len(sample) == 0:
        continue
    print(f'📄 {tipo} ({len(sample)} docs de teste):')
    for _, row in sample.iterrows():
        try:
            chunks = chunk_documento(row)
            tokens_avg = np.mean([count_tokens(c) for c in chunks]) if chunks else 0
            print(f"   [{row['ato_id'][:30]:30}] {len(chunks):4} chunks | média {tokens_avg:.0f} tokens")
        except Exception as e:
            print(f"   ❌ ERRO em {row['ato_id']}: {e}")
    print()

In [ ]:
# ============================================================
# CÉLULA 9 — Chunking completo (TODOS os documentos)
# ============================================================
# ⏱️ Tempo: ~30-40 minutos

print(f'✂️  Iniciando chunking de {len(df):,} documentos...\n')

child_chunks = []   # vai para o índice vetorial
doc_chunks_map = {} # ato_id+arquivo → lista de índices em child_chunks (para construir os pais)
stats = {'truncados': 0, 'erros': 0, 'docs_sem_chunks': 0}

for doc_idx, row in tqdm(df.iterrows(), total=len(df), desc='Chunking'):
    try:
        tipo = row['tipo_documento']
        chunks_filho = chunk_documento(row)

        if not chunks_filho:
            stats['docs_sem_chunks'] += 1
            continue

        # Aplica limite máximo por tipo (proteção contra glossários enormes)
        limite = LIMITES_CHUNKS.get(tipo, 50)
        if len(chunks_filho) > limite:
            chunks_filho = chunks_filho[:limite]
            stats['truncados'] += 1

        # Chave única do documento (alguns ato_ids têm múltiplos arquivos)
        doc_key = f"{row['ato_id']}::{row.get('arquivo_origem') or doc_idx}"
        doc_chunks_map[doc_key] = []

        for chunk_idx, chunk_text in enumerate(chunks_filho):
            chunk_text = chunk_text.strip()
            if len(chunk_text) < MIN_CHARS_CHUNK:
                continue

            contexto_juridico = extrair_contexto_juridico(
                chunk_text, str(row['ato_id']), str(row.get('titulo') or '')
            )

            child_idx = len(child_chunks)
            child_chunks.append({
                'child_id':         child_idx,
                'doc_key':          doc_key,
                'chunk_idx':        chunk_idx,
                'texto':            chunk_text,
                'tokens':           count_tokens(chunk_text),
                # Metadados para retrieval e citação
                'ato_id':           str(row['ato_id']),
                'tipo_documento':   tipo,
                'titulo':           str(row.get('titulo') or ''),
                'ementa':           (str(row.get('ementa') or ''))[:500],
                'assunto':          str(row.get('assunto') or ''),
                'situacao':         str(row.get('situacao') or ''),
                'publicacao':       str(row.get('publicacao') or '')[:10],
                'autor':            str(row.get('autor') or ''),
                'ano':              int(row['ano']),
                'contexto_juridico': contexto_juridico,
                'arquivo_origem':   str(row.get('arquivo_origem') or ''),
            })
            doc_chunks_map[doc_key].append(child_idx)

    except Exception as e:
        stats['erros'] += 1
        if stats['erros'] < 5:
            print(f"⚠️  Erro em {row.get('ato_id')}: {e}")

print(f'\n✅ Chunking concluído!')
print(f"   Total de chunks filho: {len(child_chunks):,}")
print(f"   Documentos sem chunks: {stats['docs_sem_chunks']}")
print(f"   Documentos truncados: {stats['truncados']}")
print(f"   Erros: {stats['erros']}")

In [ ]:
# ============================================================
# CÉLULA 10 — Construção dos chunks PAI
# ============================================================
# Pai = filho_anterior + filho_atual + filho_seguinte
# Limitado a PARENT_MAX_TOKENS — trunca extremos se exceder

print('👨‍👦 Construindo chunks pai...')

parent_texts = [None] * len(child_chunks)  # alinhado por child_id

for doc_key, child_ids in tqdm(doc_chunks_map.items(), desc='Pais'):
    n = len(child_ids)
    for i, cid in enumerate(child_ids):
        # Vizinhos imediatos
        prev_text = child_chunks[child_ids[i-1]]['texto'] if i > 0 else ''
        curr_text = child_chunks[cid]['texto']
        next_text = child_chunks[child_ids[i+1]]['texto'] if i < n-1 else ''

        # Concatena
        parts = [p for p in [prev_text, curr_text, next_text] if p]
        parent = '\n\n'.join(parts)

        # Trunca por tokens se exceder
        parent_tokens = count_tokens(parent)
        if parent_tokens > PARENT_MAX_TOKENS:
            # Garante que o chunk atual sempre cabe; ajusta vizinhos proporcionalmente
            curr_tokens = count_tokens(curr_text)
            available = PARENT_MAX_TOKENS - curr_tokens
            if available > 0:
                # Distribui o espaço restante entre prev e next
                half = available // 2
                tok_prev = tokenizer.encode(prev_text, add_special_tokens=False)[-half:] if prev_text else []
                tok_next = tokenizer.encode(next_text, add_special_tokens=False)[:half]   if next_text else []
                prev_truncado = tokenizer.decode(tok_prev, skip_special_tokens=True).strip()
                next_truncado = tokenizer.decode(tok_next, skip_special_tokens=True).strip()
                parts = [p for p in [prev_truncado, curr_text, next_truncado] if p]
                parent = '\n\n'.join(parts)
            else:
                parent = curr_text  # filho sozinho já estoura — usa só ele

        parent_texts[cid] = parent

# Anexa o pai a cada chunk filho
for i, p_text in enumerate(parent_texts):
    child_chunks[i]['texto_pai'] = p_text or child_chunks[i]['texto']
    child_chunks[i]['tokens_pai'] = count_tokens(child_chunks[i]['texto_pai'])

print(f'✅ Pais construídos para {len(child_chunks):,} chunks')
print(f"   Média tokens filho: {np.mean([c['tokens'] for c in child_chunks]):.1f}")
print(f"   Média tokens pai:   {np.mean([c['tokens_pai'] for c in child_chunks]):.1f}")

In [ ]:
# ============================================================
# CÉLULA 11 — Estatísticas finais
# ============================================================
df_chunks = pd.DataFrame(child_chunks)

print('📊 Distribuição dos chunks por tipo:')
for tipo, count in df_chunks['tipo_documento'].value_counts().items():
    pct = count / len(df_chunks) * 100
    print(f'   {tipo:18} {count:>8,} ({pct:5.1f}%)')

print(f'\n📏 Tamanhos:')
print(df_chunks[['tokens', 'tokens_pai']].describe().round(1).to_string())

print(f'\n💾 Tamanho em memória: {df_chunks.memory_usage(deep=True).sum() / 1e6:.1f} MB')

In [ ]:
# ============================================================
# CÉLULA 12 — Salvar chunks no Hugging Face
# ============================================================
from huggingface_hub import HfApi
from dotenv import load_dotenv
import os

load_dotenv()
HF_TOKEN = os.environ.get('HF_TOKEN', '')
HF_REPO  = 'JvPetas/aneel-legislacao'

if not SKIP_CHUNKING:
    local_path = f'{OUTPUT_DIR}/chunks_hierarquicos.parquet'
    df_chunks.to_parquet(local_path, compression='snappy', index=False)
    size_mb = os.path.getsize(local_path) / 1e6
    print(f'Salvo localmente: {local_path} ({size_mb:.1f} MB)')

    print('Enviando para Hugging Face...')
    api = HfApi(token=HF_TOKEN)
    api.upload_file(
        path_or_fileobj=local_path,
        path_in_repo='chunks_hierarquicos.parquet',
        repo_id=HF_REPO,
        repo_type='dataset',
    )
    print('chunks_hierarquicos.parquet publicado em JvPetas/aneel-legislacao')
else:
    print('Chunking foi pulado — parquet ja estava no HF.')

print('\nNOTEBOOK 1 CONCLUIDO!')
print('   Proximo: 02_embeddings_indexacao.ipynb')